# Diffusion full / LoRA 微调

[English](01_finetune.ipynb) | 简体中文

## 目标

运行独立、离线的 latent 文生图流程：校验图片/caption，训练 full 和 LoRA UNet，validation 选模，
导出、重载、生成及预览发布。Tiny UNet、VAE、CLIP 文本编码器均随机初始化，生成图片不代表质量。

## 环境准备

仓库根目录安装 `python -m pip install -e '.[diffusion,notebooks,publish]'`。
本教程使用 CPU fp32、seed 42、16×16 合成图片、两个 epoch，不下载基座或上传 Hub。
每次运行创建新的输出目录。

In [1]:
from pathlib import Path
import sys, uuid, io, contextlib, os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
from diffusers.utils import logging as diffusers_logging
diffusers_logging.set_verbosity_error()
diffusers_logging.disable_progress_bar()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "src"))
from advai_notebooks.diffusion.config import load_config
from advai_notebooks.diffusion.data import load_data, prepare_data
from advai_notebooks.diffusion.pipeline import run
from advai_notebooks.diffusion.model import DiffusionModel
from advai_notebooks.diffusion.evaluate import evaluate
from advai_notebooks.diffusion.publish import publish
RUN_DIR = ROOT / "outputs/diffusion/notebook" / uuid.uuid4().hex[:8]
def quiet(call):
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return call()
print({"device": "cpu", "epochs": 2, "training_seed": 42, "resolution": 16})

{'device': 'cpu', 'epochs': 2, 'training_seed': 42, 'resolution': 16}


## 实验步骤

### 1. 校验图片、caption 和 split

输入为 `data/sample/diffusion/samples.jsonl` 和八张原始 PPM 图片。
检查 ID、group 与跨 split 的重复解码图片。

In [2]:
full_cfg = load_config(ROOT / "configs/diffusion/smoke.yaml")
full_cfg.output_dir = str(RUN_DIR / "full")
rows, data_summary = load_data(full_cfg)
print(data_summary)

{'rows': 8, 'splits': {'train': 4, 'validation': 2, 'test': 2}, 'groups': 8, 'fingerprint': '8a37a44d4b3ce59a068edf1d6c4e2c0e84e5b5ebd450f23fdfd293d620af6ee7'}


### 2. 训练完整 UNet

VAE 与文本编码器冻结。Validation 去噪 MSE 使用固定的逐样本噪声/time draws 和 VAE mode；
最佳 checkpoint 包含 epoch 0。Test 不选模。

In [3]:
full_report = quiet(lambda: run(full_cfg))
print({"method": "full", "selected_epoch": full_report["training"]["best_epoch"],
       "baseline_test_mse": full_report["baseline_test"]["metrics"]["denoising_mse"],
       "selected_test_mse": full_report["finetuned_test"]["metrics"]["denoising_mse"]})

{'method': 'full', 'selected_epoch': 2, 'baseline_test_mse': 1.1493373215198517, 'selected_test_mse': 0.9456455558538437}


### 3. 训练 LoRA UNet

只更新 attention adapters。本样例使用 rank 4 / alpha 8，检查原生缩放参数保留。
导出携带冻结的初始 pipeline，支持离线重载。

In [4]:
lora_cfg = load_config(ROOT / "configs/diffusion/lora.yaml")
lora_cfg.output_dir = str(RUN_DIR / "lora")
lora_report = quiet(lambda: run(lora_cfg))
print({"method": "lora", "selected_epoch": lora_report["training"]["best_epoch"],
       "baseline_test_mse": lora_report["baseline_test"]["metrics"]["denoising_mse"],
       "selected_test_mse": lora_report["finetuned_test"]["metrics"]["denoising_mse"]})

{'method': 'lora', 'selected_epoch': 2, 'baseline_test_mse': 1.1493373215198517, 'selected_test_mse': 1.1469150483608246}


## 检查

### 4. 重载最佳导出并复现测试结果

两张合成图片上更小的去噪 loss 是工程观测，不证明更好的生成质量。

In [5]:
models = {}
for name, report, cfg in [("full", full_report, full_cfg), ("lora", lora_report, lora_cfg)]:
    model = quiet(lambda: DiffusionModel.load(report["export"], device="cpu"))
    actual = evaluate(model, prepare_data(rows, model, cfg), cfg, "test")
    assert actual == report["finetuned_test"]
    models[name] = model
print("Full and LoRA reload exactly reproduced the saved test reports.")

Full and LoRA reload exactly reproduced the saved test reports.


### 5. 查看固定 seed 的生成图片

Prompt 为 `a red square pattern`，seed 123，四步 DDIM。16×16 结果仅为展示放大。
随机初始化组件不能生成可靠的语义图片。

In [6]:
from IPython.display import Image as DisplayImage, display
for name, model in models.items():
    generated = model.save_images(["a red square pattern"], RUN_DIR / "preview" / name, seed=123)
    print(name)
    display(DisplayImage(filename=generated["images"][0]["path"], width=256, height=256))

full


lora


### 6. 合并 LoRA 并预览发布

合并使用副本，保留原 adapter。离线 dry-run 验证原生组件包和英文模型卡，不访问网络。

In [7]:
merged = quiet(lambda: models["lora"].save(RUN_DIR / "merged", merged=True))
assert quiet(lambda: DiffusionModel.load(merged)).method == "full"
for name, report in [("full", full_report), ("lora", lora_report)]:
    plan = quiet(lambda: publish(report["export"], "your-org/your-diffusion", dry_run=True))
    print({"format": name, "status": plan["status"], "files": len(plan["files"]),
           "private": plan["private"]})

{'format': 'full', 'status': 'dry_run', 'files': 13, 'private': True}
{'format': 'lora', 'status': 'dry_run', 'files': 14, 'private': True}


## 下一步

使用 `configs/diffusion/hf.yaml` 和明确支持的 Stable Diffusion 基座，以及 split 独立的真实图片/caption。
单独评估生成质量。中断实验从同一输出目录的 `latest` 恢复，推理 export 不含 optimizer/RNG。
CUDA/MPS 与真实预训练模型需目标硬件验证。字段、CLI 和边界见 [指南](../../docs/diffusion.zh-CN.md)。
本 Notebook 不发布模型或 Dataset。